# Sentence Segmentation

### This notebook builds upon the initial handling of utterances in the Danish Parliament.
### It includes the following:

- Sentence segmentation
- Reformatting to json object for easier handling during labeling for validation, training and inference
- Datacleaning
- Splitting into training data (including validationset to be extracted), and inference data

# Now for sentence segmentation

Implementatino of the DaCy pipeline. Due to incompatibility with newer python versions, python version 3.10.19 used. Spacy version 3.5.4 has been used, as the model card on huggingface for da_dacy_large_trf specifies spacy to be >=3.5.2,<3.6.0.


*Personal note: use env: dacy_env*


In [ ]:
import spacy
import dacy

In [ ]:
for model in dacy.models():
    print(model)

## Issues

While this downloading process should be straight forward with nlp = spacy.load("da_dacy_large_trf-0.2.0"), this does not work.
Thus, the da_dacy_large_trf-any-py3-none-any.whl file has been manually from huggingface, and the path specified
- dacy_path = os.path.abspath("C:/Users/marku/Desktop/AU/Bachelor/Data_outside_git/da_dacy_large_trf-any-py3-none-any.whl")

The file must be renamed for the pipeline to accept it
- #rename da_dacy_large_trf-any-py3-none-any.whl da_dacy_large_trf-0.2.0-py3-none-any.whl

And then manually installed
- python -m pip install da_dacy_large_trf-0.2.0-py3-none-any.whl --no-deps

Verification that spacy sees the model
- import spacy
- print(spacy.util.get_installed_models())

However, dacy still does not recognize, and since it has been build on spacy infrastructure, spacy is used for loading the model instead
- nlp = spacy.load("da_dacy_large_trf")

The name of the model can be double chekced
- print(nlp.meta["name"]) # shoudl say the above name

# Be very careful about running pip installs as the environment is very unstable, with many legacy dependencies which can override eachother

In order to use GPU for sentence segmentation, check the following
- import torch
- print(torch.cuda.is_available())
- print(torch.cuda.get_device_name(0))

Check your CUDA version
- nvidia-smi

Install matching spaCy GPU package, e.g. for CUDA 11.8:
- pip install spacy[cuda118]

or CUDA 12.x:
- pip install spacy[cuda12x]


Then initialize dacy model using spacy
- import spacy
- import dacy

- spacy.prefer_gpu()  # must be called BEFORE loading the model
- nlp = spacy.load("da_dacy_large_trf", disable=["ner", "trainable_lemmatizer", "morphologizer", "coref", "entity_linker"]) 
- print(nlp.meta["name"]) 

Confirm use of GPU
- import cupy
- print(cupy.cuda.runtime.getDeviceCount())  # should be > 0

I got a pkg-packages building wheel problem when building spacy through regular pip install on cuda version (pip install spacy[cuda13x] / pip install spacy[cuda-autodetect])
- This was fixed by going around the building wheel by doing the following:
    - For CUDA 13.0:
        - pip install cupy-cuda13x
        - Then pip install spacy==3.5.4

To check what torch version got installed (CPU only or GPU enabled)
-   import torch
    print(torch.__version__) # This line will print something along the lines of x.xx.x+cpu if CPU build is installed
    print(torch.version.cuda) # This will print None if GPU installation is not found

If torch.version.cuda is None:
- Run pip uninstall torch torchvision torchaudio -y in your environment
- Check Cuda version (In command prompt) with nvidia-smi
- Go to https://pytorch.org/get-started/locally/ To find the correct version and run the pip command in terminal from that website

In [1]:
############# RUN FIRST OR KERNELS DIE ##############

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import torch
import cupy
import spacy
from pathlib import Path

print(torch.cuda.is_available()) # Base CUDA CHECK
print(cupy.cuda.runtime.getDeviceCount()) # CUPY CUDA CHECK

spacy.require_gpu() # REUQIRE GPU GOING FORWARD
print("spaCy GPU enabled")

True
1
spaCy GPU enabled


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../src").resolve()))

from sentence_segmentation import SentenceSegmentation


merged_data_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "Data_outside_git",
                            "merged_speech_data.csv")

output_path = Path(os.path.join("..",
                           "..",
                           "..",
                           "..",
                           "Data_outside_git",
                           "full_dataset.jsonl"))

SS = SentenceSegmentation(input_path=merged_data_path, output_path=output_path)

#Confirm use of GPU
#import cupy
#print(cupy.cuda.runtime.getDeviceCount())  # should be > 0

SS.sentence_segmentation(batch_size=2)

#### OBS can also be run from terminal with the following command

python sentence_segmentation.py --inpath_to_csv <path/to/merged_data> --outpath_to_jsonl <path/for/saving/jsonl> 

Optional parameters:

--n_rows number-of-paragraphs-to-run (deafault -1)

--batch_size batch-size-for-pipe (default 2)

--n_process multiprocessing (default 1)







# Data cleaning

#### OBS former cleaning included discarding chair == True
#### The sentences matching the following characteristics will be discarded

- Less than three characters (need some guidelines here. Maybe just empty strings)
- Contains parathesis
- Remove leading/trailing whitespaces along with cases of multiple whitespaces in text




In [16]:
#set path
from pathlib import Path
import os
import json
import re
from tqdm import tqdm

json_data_path = os.path.join("..",
                              "..",
                              "..",
                              "..",
                              "Data_outside_git",
                              "full_dataset.jsonl")

json_output_path = os.path.join("..",
                              "..",
                              "..",
                              "..",
                              "Data_outside_git",
                              "filtered_full_dataset.jsonl")

In [5]:
os.path.abspath(json_data_path)

'c:\\Users\\marku\\Desktop\\AU\\Bachelor\\Data_outside_git\\full_dataset.jsonl'

In [12]:
char_limit = 5

input_path = Path(json_data_path)
output_path = Path(json_output_path)

total_lines = sum(1 for _ in open(input_path, "r", encoding="utf-8"))

with input_path.open("r", encoding="utf-8") as f_in, \
     output_path.open("w", encoding="utf-8") as f_out:
    
    for line in tqdm(f_in, total=total_lines, desc= "reading and writing lines with preprocessing"):
        entry = json.loads(line)
        text = entry.get("text", "")
        
        if (len(text.strip()) > char_limit) and ("(" not in text) and (")" not in text):
            #remove multiple following whitespaces
            text = re.sub(r'\s+', ' ', text).strip()
            entry["text"] = text
            f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
        
        else:
            pass

reading and writing lines with preprocessing: 100%|██████████| 6553133/6553133 [02:43<00:00, 40009.77it/s]


In [17]:
#Total lines in preprocessed file

total_lines_out = sum(1 for _ in open(output_path, "r", encoding="utf-8"))

print(f"total number of lines in processed data: {total_lines_out}")

total number of lines in processed data: 5598994


# Test-Train-Split


#### However, validation set will be drawn from the training data due to the need for for prelimenary blame detection allowing computationally assisted lableing and artificially inflated probability of blame in validationset


In [21]:
import json
import random
from pathlib import Path
from tqdm import tqdm

#setting paths
input_path = Path(json_output_path)

train_path = Path(os.path.join("..",
                              "..",
                              "..",
                              "..",
                              "Data_outside_git",
                              "train_validation.jsonl"))

inference_path = Path(os.path.join("..",
                                   "..",
                                   "raw_data",
                               "inference",
                               "inference_data.jsonl"))

train_sample_size = 500500 #OBS set as desired size of train/test/validation set
random.seed(42)

# First pass: count total lines
with input_path.open("r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)

# Randomly select indices for train/val/test
train_indices = set(random.sample(range(total_lines), train_sample_size))

# Second pass: write to respective files
with input_path.open("r", encoding="utf-8") as f_in, \
     train_path.open("w", encoding="utf-8") as f_train, \
     inference_path.open("w", encoding="utf-8") as f_inf:

    for i, line in enumerate(tqdm(f_in, total=total_lines)):
        if i in train_indices:
            f_train.write(line)
        else:
            f_inf.write(line)

print(f"Train/val/test: {train_sample_size} lines")
print(f"Inference: {total_lines - train_sample_size} lines")

100%|██████████| 5598994/5598994 [00:23<00:00, 236640.61it/s]

Train/val/test: 500500 lines
Inference: 5098494 lines
